In [ ]:
import pandas as pdimport jsonimport osfrom sklearn.neighbors import NearestNeighborsfrom scipy.sparse import csr_matrixfrom collections import defaultdictimport numpy as np

# Spotify Song Recommendation using KNNThis notebook implements a K-Nearest Neighbors algorithm for recommending songs based on user playlists.

## Data Loading

In [ ]:
def read_slices(file_path):    try:        with open(file_path, 'r') as f:            data = json.load(f)            return data    except Exception as e:        print(f'Error reading file: {e}')        return Nonejsondata = read_slices('data/mpd.slice.0-999.json')

## Data Preprocessing

In [ ]:
def extract_tracks(jsondata):    if 'playlists' not in jsondata.keys():        raise KeyError('Key not found')    data = []    for i in range(len(jsondata['playlists'])):        for track in jsondata['playlists'][i]['tracks']:            data.append([i, track['track_uri'][14:]])    return datatrack_data = extract_tracks(jsondata)

## Encoding Tracks

In [ ]:
def encode_tracks(interactions):    track_to_idx = {}    idx_to_track = {}    track_counter = 0    encoded_interactions = []    for playlist_id, track_uri in interactions:        if track_uri not in track_to_idx:            track_to_idx[track_uri] = track_counter            idx_to_track[track_counter] = track_uri            track_counter += 1        encoded_interactions.append((playlist_id, track_to_idx[track_uri]))    return encoded_interactions, track_to_idx, idx_to_trackencoded_track_data, track_to_id, id_to_track = encode_tracks(track_data)

## Building the Sparse Matrix

In [ ]:
def build_matrix(encoded_interactions, track_to_idx):    rows = []    cols = []    data = []    for p_id, t_idx in encoded_interactions:        rows.append(p_id)        cols.append(t_idx)        data.append(1)    X = csr_matrix((data, (rows, cols)), shape=(max(rows) + 1, len(track_to_idx)))    return XX = build_matrix(encoded_track_data, track_to_id)

## Training the KNN Model

In [ ]:
knn_model = NearestNeighbors(n_neighbors=5, metric='cosine', algorithm='brute')knn_model.fit(X.T)

## Making Recommendations

In [ ]:
def recommend_tracks(seed_track_id, knn_model, track_to_id, id_to_track):    seed_idx = track_to_id[seed_track_id]    distances, neighbors = knn_model.kneighbors(X.T[seed_idx], n_neighbors=10)    recommended_tracks = [id_to_track[idx] for idx in neighbors[0]]    return recommended_tracksseed_track_id = list(track_to_id.keys())[0]  # Example seed trackrecommended = recommend_tracks(seed_track_id, knn_model, track_to_id, id_to_track)recommended

## Evaluation

In [ ]:
def evaluate_recommendations(recommended, ground_truth):    precision = sum(1 for r in recommended if r in ground_truth) / len(recommended)    return precisionground_truth = ['example_track_id_1', 'example_track_id_2']  # Replace with actual ground truthprecision = evaluate_recommendations(recommended, ground_truth)precision